In [1]:

import pandas as pd
import numpy as np

print("WasteWise-AI Feature Engineering Started!")


WasteWise-AI Feature Engineering Started!


In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

data_folder = Path(
project_root = Path.cwd().resolve()
while not (project_root / "ml" / "data" / "raw").is_dir():
    if project_root.parent == project_root:
        raise FileNotFoundError("Could not find the WasteWise-AI project root.")
    project_root = project_root.parent
data_folder = project_root / "ml" / "data" / "raw"
)

inventory = pd.read_csv(
    data_folder / "perishable_inventory (1).csv"
)

batches = pd.read_csv(
    data_folder / "inventory_batches.csv"
)

products = pd.read_csv(
    data_folder / "products.csv"
)

discounts = pd.read_csv(
    data_folder / "discount_scenarios_simulated.csv"
)

print("All 4 datasets loaded successfully!")

print("Inventory:", inventory.shape)
print("Batches:", batches.shape)
print("Products:", products.shape)
print("Discounts:", discounts.shape)


All 4 datasets loaded successfully!
Inventory: (1800, 12)
Batches: (2000, 16)
Products: (10, 6)
Discounts: (10000, 8)


In [3]:

print("INVENTORY COLUMNS:")
print(inventory.columns.tolist())

print("\nBATCHES COLUMNS:")
print(batches.columns.tolist())

print("\nPRODUCTS COLUMNS:")
print(products.columns.tolist())

print("\nDISCOUNTS COLUMNS:")
print(discounts.columns.tolist())


INVENTORY COLUMNS:
['product_id', 'date', 'category', 'opening_stock', 'units_received', 'units_sold', 'unit_price', 'discount_pct', 'days_to_expiry', 'temperature_c', 'waste_units', 'stockout_flag']

BATCHES COLUMNS:
['batch_id', 'product_id', 'category', 'received_at', 'snapshot_date', 'expiry_date', 'initial_quantity', 'quantity_remaining', 'days_to_expiry', 'avg_daily_sales_7d', 'expected_demand_before_expiry', 'unit_price', 'unit_cost', 'discount_pct', 'expiry_risk_label', 'waste_units_actual_simulated']

PRODUCTS COLUMNS:
['product_id', 'product_name', 'category', 'shelf_life_days', 'selling_price', 'unit_cost']

DISCOUNTS COLUMNS:
['batch_id', 'product_id', 'days_to_expiry', 'quantity_remaining', 'discount_pct', 'estimated_units_sold_simulated', 'estimated_revenue_simulated', 'estimated_gross_margin_simulated']


In [4]:

datasets = {
    "Inventory": inventory,
    "Batches": batches,
    "Products": products,
    "Discounts": discounts
}

for name, df in datasets.items():
    print(f"\n{name}")

    date_columns = [
        col for col in df.columns
        if any(word in col.lower()
               for word in ["date", "expiry", "expire"])
    ]

    quantity_columns = [
        col for col in df.columns
        if any(word in col.lower()
               for word in ["quantity", "stock", "sold", "sales"])
    ]

    print("Date Columns:", date_columns)
    print("Quantity Columns:", quantity_columns)



Inventory
Date Columns: ['date', 'days_to_expiry']
Quantity Columns: ['opening_stock', 'units_sold', 'stockout_flag']

Batches
Date Columns: ['snapshot_date', 'expiry_date', 'days_to_expiry', 'expected_demand_before_expiry', 'expiry_risk_label']
Quantity Columns: ['initial_quantity', 'quantity_remaining', 'avg_daily_sales_7d']

Products
Date Columns: []
Quantity Columns: []

Discounts
Date Columns: ['days_to_expiry']
Quantity Columns: ['quantity_remaining', 'estimated_units_sold_simulated']


In [5]:

names = list(datasets.keys())

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        common = sorted(
            set(datasets[names[i]].columns)
            & set(datasets[names[j]].columns)
        )

        print(f"{names[i]} + {names[j]}: {common}")


Inventory + Batches: ['category', 'days_to_expiry', 'discount_pct', 'product_id', 'unit_price']
Inventory + Products: ['category', 'product_id']
Inventory + Discounts: ['days_to_expiry', 'discount_pct', 'product_id']
Batches + Products: ['category', 'product_id', 'unit_cost']
Batches + Discounts: ['batch_id', 'days_to_expiry', 'discount_pct', 'product_id', 'quantity_remaining']
Products + Discounts: ['product_id']


In [6]:

print("Inventory Columns:")
print(inventory.columns.tolist())

print("\nBatches Columns:")
print(batches.columns.tolist())

# Find possible expiry columns
for name, df in {
    "Inventory": inventory,
    "Batches": batches
}.items():
    expiry_cols = [
        col for col in df.columns
        if "expir" in col.lower()
        or "best_before" in col.lower()
    ]

    print(name, "Expiry Columns:", expiry_cols)


Inventory Columns:
['product_id', 'date', 'category', 'opening_stock', 'units_received', 'units_sold', 'unit_price', 'discount_pct', 'days_to_expiry', 'temperature_c', 'waste_units', 'stockout_flag']

Batches Columns:
['batch_id', 'product_id', 'category', 'received_at', 'snapshot_date', 'expiry_date', 'initial_quantity', 'quantity_remaining', 'days_to_expiry', 'avg_daily_sales_7d', 'expected_demand_before_expiry', 'unit_price', 'unit_cost', 'discount_pct', 'expiry_risk_label', 'waste_units_actual_simulated']
Inventory Expiry Columns: ['days_to_expiry']
Batches Expiry Columns: ['expiry_date', 'days_to_expiry', 'expected_demand_before_expiry', 'expiry_risk_label']


In [7]:

# Work on a copy of the batch dataset
batches_features = batches.copy()

expiry_cols = [
    col for col in batches_features.columns
    if "expir" in col.lower()
    or "best_before" in col.lower()
]

if expiry_cols:
    expiry_col = expiry_cols[0]

    batches_features[expiry_col] = pd.to_datetime(
        batches_features[expiry_col],
        errors="coerce"
    )

    # Days until expiry as of today
    today = pd.Timestamp.today().normalize()

    batches_features["days_until_expiry"] = (
        batches_features[expiry_col] - today
    ).dt.days

    display(
        batches_features[
            [expiry_col, "days_until_expiry"]
        ].head()
    )
else:
    print("No expiry column found.")


,expiry_date,days_until_expiry
0,2026-08-21,-68
1,2026-07-24,-96
2,2026-08-23,-66
3,2026-08-25,-64
4,2026-07-30,-90


In [8]:

if "days_until_expiry" in batches_features.columns:

    days = batches_features["days_until_expiry"]

    batches_features["expiry_urgency"] = np.select(
        [
            days < 0,
            days <= 3,
            days <= 7
        ],
        [
            "Expired",
            "High",
            "Medium"
        ],
        default="Low"
    )

    batches_features.loc[
        days.isna(), "expiry_urgency"
    ] = "Unknown"

    print(batches_features["expiry_urgency"].value_counts())

    display(batches_features.head())
else:
    print("Expiry feature not available.")


expiry_urgency
Expired    1981
High          9
Medium        6
Low           4
Name: count, dtype: int64


,batch_id,product_id,category,received_at,snapshot_date,expiry_date,initial_quantity,quantity_remaining,days_to_expiry,avg_daily_sales_7d,expected_demand_before_expiry,unit_price,unit_cost,discount_pct,expiry_risk_label,waste_units_actual_simulated,days_until_expiry,expiry_urgency
0,5001,1006,Produce,2026-08-14,2026-08-16,2026-08-21,140,72,5,23.38,117,45.0,28.0,5,1,64,-68,Expired
1,5002,1006,Produce,2026-07-17,2026-07-23,2026-07-24,140,123,1,36.86,37,45.0,28.0,10,1,31,-96,Expired
2,5003,1001,Dairy,2026-08-18,2026-08-20,2026-08-23,22,3,3,12.67,38,32.0,24.0,10,1,3,-66,Expired
3,5004,1003,Bakery,2026-08-21,2026-08-26,2026-08-25,141,141,-1,14.52,0,40.0,26.0,10,1,114,-64,Expired
4,5005,1007,Produce,2026-07-16,2026-07-19,2026-07-30,139,52,11,13.09,144,120.0,78.0,20,1,34,-90,Expired


In [9]:

output_folder = project_root / "ml" / "data" / "processed"

output_folder.mkdir(parents=True, exist_ok=True)

batches_features.to_csv(
    output_folder / "batches_features.csv",
    index=False
)

print("Feature engineering dataset saved successfully!")


Feature engineering dataset saved successfully!
